In [1]:
import numpy as np
import os
import PIL
import PIL.Image
import tensorflow as tf
import keras
from keras.layers import Conv2D, Flatten, Dense, MaxPooling2D, Dropout
from keras.applications.resnet50 import ResNet50
from keras.applications.resnet50 import preprocess_input, decode_predictions
from keras import datasets, layers, models
from keras.models import Sequential
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
import io
import cv2
from sklearn.utils import class_weight
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, auc, accuracy_score, roc_curve

In [2]:
batch_size = 64
img_height = 140
img_width = 140

In [3]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    'C:\\Users\\User\\Desktop\\emotion\\dataset_affectnet\\Train',
    subset ="training",
    validation_split = 0.2,
    seed = 123,
    batch_size = batch_size,
    image_size =(img_height, img_width),
)

Found 14549 files belonging to 7 classes.
Using 11640 files for training.


In [4]:
val_ds= tf.keras.utils.image_dataset_from_directory(
    'C:\\Users\\User\\Desktop\\emotion\\dataset_affectnet\\Train',
    seed =123,
    subset = "validation",
    validation_split = 0.2,
    image_size = (img_height, img_width),
    batch_size = batch_size,
)

Found 14549 files belonging to 7 classes.
Using 2909 files for validation.


In [5]:
y_train = np.concatenate([y for x, y in train_ds], axis=0)

In [9]:
y_train

array([3, 5, 2, ..., 0, 0, 2], shape=(11640,), dtype=int32)

In [7]:
classes = np.unique(y_train)


In [10]:
classes

array([0, 1, 2, 3, 4, 5, 6], dtype=int32)

In [11]:
weights = class_weight.compute_class_weight(class_weight='balanced', classes=classes, y=y_train)

In [12]:
weights

array([1.36973406, 1.71782763, 1.3834086 , 0.87935333, 0.75344683,
       0.6770591 , 0.97700185])

In [13]:
weights_dict = dict(enumerate(weights))

In [14]:
weights_dict

{0: np.float64(1.3697340550717816),
 1: np.float64(1.717827626918536),
 2: np.float64(1.3834086047064416),
 3: np.float64(0.8793533277933067),
 4: np.float64(0.7534468250372193),
 5: np.float64(0.677059097254537),
 6: np.float64(0.9770018465670639)}

In [ ]:
import matplotlib.pyplot as plt

# Take one batch from dataset
for images, labels in train_ds.take(1):
    plt.figure(figsize=(10,10))

    for i in range(9):   # show 9 images
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(labels[i].numpy())
        plt.axis("off")

    plt.show()

In [ ]:
train_ds = train_ds.map(lambda x, y:(preprocess_input(tf.cast(x, tf.float32)), y))
val_ds = val_ds_1.map(lambda x, y:(preprocess_input(tf.cast(x, tf.float32)), y))
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(buffer_size = AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size = AUTOTUNE)

In [ ]:
data_aug =tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomZoom(0.1),
    
])

In [ ]:
base_model = keras.applications.ResNet50(
    weights = 'imagenet',
    input_shape=(140, 140, 3),
    include_top= False)

base_model.trainable = False #this freezes the base model.

inputs = keras.Input(shape=(140, 140, 3))
x= data_aug(inputs)
x = base_model(x, training = False)
x = keras.layers.GlobalAveragePooling2D() (x)
x = keras.layers.Dense(256, activation='relu') (x)
x = layers.Dropout(0.5)(x)
outputs = keras.layers.Dense(7, activation="softmax") (x)
model = keras.Model(inputs, outputs)

In [ ]:
base_model.trainable = True
base_learning_rate = 0.0001
print("Number of layers in a base-model:", len(base_model.layers))
#fine tune from this layer onwards
fine_tune_from= 100
#freezing all the layers from 0 to 99
for layer in base_model.layers[:fine_tune_from]:
    layer.trainable= False

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(), 
    loss = 'sparse_categorical_crossentropy', metrics=['accuracy'])

In [ ]:
history1 = model.fit(train_ds, validation_data=val_ds, epochs = 10, class_weight=weights_dict)